In [9]:
### Import des modules
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [10]:
# Import des fichiers
data_sirh = pd.read_csv('../data/raw/extrait_sirh.csv', sep=',', na_values=[''], quotechar='"')
data_eval = pd.read_csv('../data/raw/extrait_eval.csv', sep=',', na_values=[''], quotechar='"')
data_sondage = pd.read_csv('../data/raw/extrait_sondage.csv', sep=',', na_values=[''], quotechar='"')

In [11]:
# La colonne eval_number va nous servir pour la jointure des fichiers E_employee_id
data_eval['id_employee'] = data_eval['eval_number'].apply(lambda x : x.split('E_')[1])
data_eval['id_employee'] = data_eval['id_employee'].astype(int)
data_eval.drop(columns=['eval_number'], inplace=True)

# On renomme juste la colonne code_sonadage qui contient l'employee_id
data_sondage.rename(columns={'code_sondage': 'id_employee'}, inplace=True)

In [12]:
# Fusion des fichiers
data = data_sirh.merge(data_eval, how='inner')
data = data.merge(data_sondage, how='inner')

In [13]:
cols_to_remove = [
    'id_employee',
    'nombre_heures_travailless',
    'augementation_salaire_precedente',
    'nombre_employee_sous_responsabilite',
    'ayant_enfants'
]

data.drop(columns=cols_to_remove, inplace=True)

In [14]:
data['genre'] = data['genre'].map({'F' : 0, 'M': 1})
data['a_quitte_l_entreprise'] = data['a_quitte_l_entreprise'].map({'Non' : 0, 'Oui': 1})
data['heure_supplementaires'] = data['heure_supplementaires'].map({'Non': 0, 'Oui': 1})
data['frequence_deplacement'] = data['frequence_deplacement'].map({'Aucun': 1, 'Occasionnel' : 1.5, 'Frequent' : 2})

In [15]:
from pathlib import Path
import os

base_dir = Path(os.getcwd()).resolve().parent

if not os.path.exists(base_dir / 'data/rafined/'):
    os.mkdir(base_dir / 'data/rafined/')

data.to_csv(base_dir / 'data/rafined/employees.csv', index=False)